<a href="https://colab.research.google.com/github/fvangool/Deep-Learning-Specialization-Coursera/blob/main/Feature_engineering_v_21.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Feature Engineering — Central Pipeline
PS S6E4 Irrigation Need Prediction
=====================================
Single source of truth for all feature engineering.
Run this notebook ONCE before training any model.

Outputs (saved to OUT_DIR):
  train_engineered_{FE_VERSION}.parquet   — train + original, all features
  test_engineered_{FE_VERSION}.parquet    — test, all features
  feature_metadata_{FE_VERSION}.json     — column lists, checksums, version

Model notebooks load the parquet directly — no feature engineering needed.

FE_VERSION history:
  v21 — initial centralised version
        + dual_boundary_proximity, boundary_proximity
        + prior_irrigation_intensity, high_prior_irrigation
        + humidity_at_boundary, vpd_at_boundary
        + margin_Medium_wins, formula_predicts_medium (via local score vars)
        - score_High, score_Low, score_Medium REMOVED (zero variance on hard rows)
        - margin_High_Medium, margin_Low_Medium REMOVED (redundant with magic_score)
"""

# ============================================================
# INSTALL / IMPORTS
# ============================================================
import gc
import os
import json
import time
import hashlib
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

print("Feature Engineering Pipeline")
print("=" * 60)

# ============================================================
# SECTION 0 — CONFIGURATION
# ============================================================
TRAIN_PATH    = "/content/drive/MyDrive/irrigation_need_v15/train.csv"
TEST_PATH     = "/content/drive/MyDrive/irrigation_need_v15/test.csv"
ORIGINAL_PATH = "/content/drive/MyDrive/irrigation_need_v15/irrigation_prediction.csv"
OUT_DIR       = "/content/drive/MyDrive/irrigation_need_v15/"
HARD_IDX_PATH = "/content/drive/MyDrive/irrigation_need_v15/hard_example_indices.npy"

# Bump this every time engineer_features() changes — model notebooks
# assert this version matches what they expect.
FE_VERSION = "v21"

TARGET         = "Irrigation_Need"
TARGET_MAPPING = {"Low": 0, "Medium": 1, "High": 2}

NUMS = [
    "Soil_pH", "Soil_Moisture", "Organic_Carbon",
    "Electrical_Conductivity", "Temperature_C", "Humidity",
    "Rainfall_mm", "Sunlight_Hours", "Wind_Speed_kmh",
    "Field_Area_hectare", "Previous_Irrigation_mm",
]
CATS = [
    "Soil_Type", "Crop_Type", "Crop_Growth_Stage", "Season",
    "Irrigation_Type", "Water_Source", "Mulching_Used", "Region",
]

SOIL_THRESH = 25
RAIN_THRESH = 300
TEMP_THRESH = 30
WIND_THRESH = 10

LOGIT_COEFS = {
    "Low": {
        "intercept": 16.3173, "soil_lt_25": -11.0237, "temp_gt_30": -5.8559,
        "rain_lt_300": -10.8500, "wind_gt_10": -5.8284,
        "Flowering": -5.4155, "Harvest": 5.5073, "Sowing": 5.2299,
        "Vegetative": -5.4617, "Mulch_No": -3.0014, "Mulch_Yes": 2.8613,
    },
    "Medium": {
        "intercept": 4.6524, "soil_lt_25": 0.3290, "temp_gt_30": -0.0204,
        "rain_lt_300": 0.1542, "wind_gt_10": 0.0841,
        "Flowering": 0.3586, "Harvest": -0.1348, "Sowing": -0.3547,
        "Vegetative": 0.3334, "Mulch_No": 0.1883, "Mulch_Yes": 0.0142,
    },
    "High": {
        "intercept": -20.9697, "soil_lt_25": 10.6947, "temp_gt_30": 5.8763,
        "rain_lt_300": 10.6958, "wind_gt_10": 5.7444,
        "Flowering": 5.0569, "Harvest": -5.3725, "Sowing": -4.8752,
        "Vegetative": 5.1283, "Mulch_No": 2.8131, "Mulch_Yes": -2.8755,
    },
}

# All new features added beyond the raw NUMS/CATS columns.
# This list is saved to metadata and checked by model notebooks.
NEW_FEATURES = [
    # ── Binary threshold features ──────────────────────────
    "soil_lt_25", "rain_lt_300", "temp_gt_30", "wind_gt_10",
    "is_harvest", "is_sowing", "mulching_yes",
    # ── Magic score family ─────────────────────────────────
    "magic_score", "dist_boundary_0", "dist_boundary_3", "dist_boundary_min",
    # ── Continuous threshold distances ────────────────────
    "soil_dist_25", "rain_dist_300", "temp_dist_30", "wind_dist_10",
    # ── Decimal digit features ─────────────────────────────
    *[f"{col}_dec"  for col in NUMS],
    *[f"{col}_dec2" for col in NUMS],
    # ── Logit scores ───────────────────────────────────────
    "logit_Low", "logit_Medium", "logit_High",
    # ── Domain interaction features ────────────────────────
    "ET_Proxy", "Total_Water_Input", "Moisture_Deficit", "Irrigation_Ratio",
    "Evap_Stress", "Wind_ET", "Net_Water_Need", "VPD_Proxy", "Heat_Stress",
    "Dryness_Index", "Aridity_Index", "Drought_Risk", "Soil_Health",
    "Salinity_Risk", "pH_Deviation", "Moisture_Retention", "EC_pH_Interaction",
    "Irrig_Per_Ha", "Rain_Per_Ha", "Area_Log", "Water_Per_Ha",
    "Mulch_Flag", "Mulch_Moisture", "Mulch_ET_Saving",
    "Moisture_ET_Ratio", "Rain_ET_Balance", "Sunlight_Temp_Ratio",
    "Water_Stress_Index",
    "log_Rainfall_mm", "log_Previous_Irrigation_mm",
    "log_Field_Area_hectare", "log_Wind_Speed_kmh",
    # ── Boundary zone / Medium disambiguation ──────────────
    "soil_moisture_deficit", "rainfall_deficit", "temp_excess", "wind_excess",
    "magic_score_soft", "at_magic_boundary", "flowering_at_boundary",
    "warm_dry_stress",
    # ── Score-derived (via local vars, no raw scores in df) ─
    "margin_Medium_wins", "formula_predicts_medium",
    # ── NEW v21: hard-case targeted features ───────────────
    "boundary_proximity", "dual_boundary_proximity",
    "prior_irrigation_intensity", "high_prior_irrigation",
    "humidity_at_boundary", "vpd_at_boundary",
    "sandy_at_boundary", "canal_at_boundary",
]

print(f"\nFE_VERSION  : {FE_VERSION}")
print(f"TRAIN_PATH  : {TRAIN_PATH}")
print(f"TEST_PATH   : {TEST_PATH}")
print(f"OUTPUT DIR  : {OUT_DIR}")
print(f"New features: {len(NEW_FEATURES)}")

# ============================================================
# SECTION 1 — FEATURE ENGINEERING FUNCTIONS
# ============================================================
def compute_logit_scores(df: pd.DataFrame) -> pd.DataFrame:
    soil  = (df["Soil_Moisture"]  < SOIL_THRESH).astype(float)
    temp  = (df["Temperature_C"]  > TEMP_THRESH).astype(float)
    rain  = (df["Rainfall_mm"]    < RAIN_THRESH).astype(float)
    wind  = (df["Wind_Speed_kmh"] > WIND_THRESH).astype(float)
    stage = df["Crop_Growth_Stage"].astype(str)
    mulch = df["Mulching_Used"].astype(str)

    binary = {
        "soil_lt_25": soil, "temp_gt_30": temp,
        "rain_lt_300": rain, "wind_gt_10": wind,
    }
    flags = {
        "Flowering":  (stage == "Flowering").astype(float),
        "Harvest":    (stage == "Harvest").astype(float),
        "Sowing":     (stage == "Sowing").astype(float),
        "Vegetative": (stage == "Vegetative").astype(float),
        "Mulch_No":   (mulch == "No").astype(float),
        "Mulch_Yes":  (mulch == "Yes").astype(float),
    }
    for cls_name, coefs in LOGIT_COEFS.items():
        df[f"logit_{cls_name}"] = (
            coefs["intercept"]
            + sum(coefs[k] * v for k, v in binary.items())
            + sum(coefs[k] * v for k, v in flags.items())
        )
    return df


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # ── Binary threshold features ─────────────────────────────────
    df["soil_lt_25"]   = (df["Soil_Moisture"]     < SOIL_THRESH).astype(int)
    df["rain_lt_300"]  = (df["Rainfall_mm"]        < RAIN_THRESH).astype(int)
    df["temp_gt_30"]   = (df["Temperature_C"]      > TEMP_THRESH).astype(int)
    df["wind_gt_10"]   = (df["Wind_Speed_kmh"]     > WIND_THRESH).astype(int)
    df["is_harvest"]   = (df["Crop_Growth_Stage"] == "Harvest").astype(int)
    df["is_sowing"]    = (df["Crop_Growth_Stage"] == "Sowing").astype(int)
    df["mulching_yes"] = (df["Mulching_Used"]      == "Yes").astype(int)

    # ── Magic score + boundary distances ─────────────────────────
    high_s = (2*df["soil_lt_25"] + 2*df["rain_lt_300"]
              + df["temp_gt_30"] + df["wind_gt_10"])
    low_s  = 2*df["is_harvest"] + 2*df["is_sowing"] + df["mulching_yes"]
    df["magic_score"]       = (high_s - low_s).astype(int)
    df["dist_boundary_0"]   = np.abs(df["magic_score"] - 0)
    df["dist_boundary_3"]   = np.abs(df["magic_score"] - 3)
    df["dist_boundary_min"] = np.minimum(
        df["dist_boundary_0"], df["dist_boundary_3"]
    )

    # ── Continuous threshold distances ────────────────────────────
    df["soil_dist_25"]  = df["Soil_Moisture"]  - SOIL_THRESH
    df["rain_dist_300"] = df["Rainfall_mm"]    - RAIN_THRESH
    df["temp_dist_30"]  = df["Temperature_C"]  - TEMP_THRESH
    df["wind_dist_10"]  = df["Wind_Speed_kmh"] - WIND_THRESH

    # ── Decimal digit features (1st and 2nd decimal place) ───────
    for col in NUMS:
        vals = df[col].values
        frac = vals - np.floor(vals)
        df[f"{col}_dec"]  = np.floor(frac * 10).astype(int)
        df[f"{col}_dec2"] = (np.floor(frac * 100) % 10).astype(int)

    # ── Logit scores from LOGIT_COEFS ────────────────────────────
    df = compute_logit_scores(df)

    # ── Domain interaction features ───────────────────────────────
    sm  = df["Soil_Moisture"];   tmp = df["Temperature_C"]
    rf  = df["Rainfall_mm"];     hum = df["Humidity"]
    sun = df["Sunlight_Hours"];  wnd = df["Wind_Speed_kmh"]
    oc  = df["Organic_Carbon"];  ec  = df["Electrical_Conductivity"]
    ph  = df["Soil_pH"];         fa  = df["Field_Area_hectare"]
    pi  = df["Previous_Irrigation_mm"]

    df["ET_Proxy"]            = tmp * sun / (hum + 1)
    df["Total_Water_Input"]   = rf + pi
    df["Moisture_Deficit"]    = 100 - sm
    df["Irrigation_Ratio"]    = pi / (rf + 1)
    df["Evap_Stress"]         = tmp * wnd / (hum + 1)
    df["Wind_ET"]             = wnd * (1 - hum / 100)
    df["Net_Water_Need"]      = df["ET_Proxy"] - rf / 10
    df["VPD_Proxy"]           = tmp * (1 - hum / 100)
    df["Heat_Stress"]         = tmp * (100 - hum) / 100
    df["Dryness_Index"]       = tmp * sun / (rf + 1)
    df["Aridity_Index"]       = rf / (df["ET_Proxy"] + 0.1)
    df["Drought_Risk"]        = df["Dryness_Index"] * df["Moisture_Deficit"] / 100
    df["Soil_Health"]         = oc * sm / (ec + 0.1)
    df["Salinity_Risk"]       = ec * tmp / (rf + 1)
    df["pH_Deviation"]        = np.abs(ph - 6.5)
    df["Moisture_Retention"]  = sm * oc
    df["EC_pH_Interaction"]   = ec * ph
    df["Irrig_Per_Ha"]        = pi / (fa + 0.1)
    df["Rain_Per_Ha"]         = rf / (fa + 0.1)
    df["Area_Log"]            = np.log1p(fa)
    df["Water_Per_Ha"]        = df["Total_Water_Input"] / (fa + 0.1)

    mulch_flag            = df["mulching_yes"].astype(float)
    df["Mulch_Flag"]      = mulch_flag
    df["Mulch_Moisture"]  = sm * (1 + 0.5 * mulch_flag)
    df["Mulch_ET_Saving"] = df["ET_Proxy"] * (1 - 0.3 * mulch_flag)

    df["Moisture_ET_Ratio"]   = sm / (df["ET_Proxy"] + 0.1)
    df["Rain_ET_Balance"]     = rf - df["ET_Proxy"] * 2
    df["Sunlight_Temp_Ratio"] = sun / (tmp + 1)
    df["Water_Stress_Index"]  = (
        (df["ET_Proxy"] - df["Total_Water_Input"] / 10) / (sm + 1)
    )

    for col in ["Rainfall_mm", "Previous_Irrigation_mm",
                "Field_Area_hectare", "Wind_Speed_kmh"]:
        df[f"log_{col}"] = np.log1p(df[col])

    # ── Boundary zone features (Medium disambiguation) ────────────
    df["soil_moisture_deficit"] = np.maximum(
        0, SOIL_THRESH - df["Soil_Moisture"]
    ).astype(np.float32)
    df["rainfall_deficit"] = np.maximum(
        0, RAIN_THRESH - df["Rainfall_mm"]
    ).astype(np.float32)
    df["temp_excess"] = np.maximum(
        0, df["Temperature_C"] - TEMP_THRESH
    ).astype(np.float32)
    df["wind_excess"] = np.maximum(
        0, df["Wind_Speed_kmh"] - WIND_THRESH
    ).astype(np.float32)

    df["magic_score_soft"] = (
        2 * df["soil_moisture_deficit"] / SOIL_THRESH +
        2 * df["rainfall_deficit"]      / RAIN_THRESH +
        df["temp_excess"]               / WIND_THRESH +
        df["wind_excess"]               / WIND_THRESH
    ).astype(np.float32)

    df["at_magic_boundary"] = df["magic_score"].isin(
        [0, 3, 4, 5]
    ).astype(np.int8)

    df["flowering_at_boundary"] = (
        (df["Crop_Growth_Stage"].isin(["Flowering", "Vegetative"])) &
        (df["magic_score"].isin([0, 3]))
    ).astype(np.int8)

    df["warm_dry_stress"] = (
        df["temp_excess"] * df["soil_moisture_deficit"]
    ).astype(np.float32)

    # ── Score reconstruction (local vars only — not stored in df) ─
    # Storing score_High/Low/Medium directly caused zero-variance on
    # hard rows (score_Medium = 2.0 constant). Derived features only.
    s  = df["soil_lt_25"].astype(float)
    t  = df["temp_gt_30"].astype(float)
    r  = df["rain_lt_300"].astype(float)
    w  = df["wind_gt_10"].astype(float)
    mu = df["mulching_yes"].astype(float)
    fl = (df["Crop_Growth_Stage"] == "Flowering").astype(float)
    ha = df["is_harvest"].astype(float)
    so = df["is_sowing"].astype(float)
    ve = (df["Crop_Growth_Stage"] == "Vegetative").astype(float)

    _score_High   = ( 4*s + 2*t + 4*r + 2*w - 2*mu - 5*fl - 9*ha - 9*so - 5*ve)
    _score_Low    = (-4*s - 2*t - 4*r - 2*w + 2*mu + 3*fl + 7*ha + 7*so + 3*ve)
    _score_Medium = (2*fl + 2*ha + 2*so + 2*ve)

    df["margin_Medium_wins"] = (
        _score_Medium - np.maximum(_score_High, _score_Low)
    ).astype(np.float32)
    df["formula_predicts_medium"] = (
        (_score_Medium > _score_High) & (_score_Medium > _score_Low)
    ).astype(np.int8)

    # ── NEW v21: Hard-case targeted features ──────────────────────
    # boundary_proximity: joint proximity to both soil AND rain thresholds
    df["boundary_proximity"] = (
        np.abs(df["soil_dist_25"])  / SOIL_THRESH  *
        np.abs(df["rain_dist_300"]) / RAIN_THRESH
    ).astype(np.float32)

    # dual_boundary_proximity: binary flag for rows near BOTH thresholds
    df["dual_boundary_proximity"] = (
        (np.abs(df["soil_dist_25"])  < 5) &
        (np.abs(df["rain_dist_300"]) < 50)
    ).astype(np.int8)

    # prior_irrigation_intensity: prior irrigation relative to rainfall
    df["prior_irrigation_intensity"] = (
        pi / (rf + 1)
    ).astype(np.float32)

    # high_prior_irrigation: binary flag (mean of hard Medium rows = 64.9)
    df["high_prior_irrigation"] = (
        df["Previous_Irrigation_mm"] > 60
    ).astype(np.int8)

    # humidity_at_boundary: humidity signal active only at boundary zone
    df["humidity_at_boundary"] = (
        df["Humidity"] * df["at_magic_boundary"]
    ).astype(np.float32)

    # vpd_at_boundary: VPD signal active only at boundary zone
    df["vpd_at_boundary"] = (
        df["VPD_Proxy"] * df["at_magic_boundary"]
    ).astype(np.float32)

    # sandy_at_boundary: highest hard-rate soil type at boundary
    df["sandy_at_boundary"] = (
        (df["Soil_Type"] == "Sandy") &
        df["magic_score"].isin([0, 3])
    ).astype(np.int8)

    # canal_at_boundary: highest hard-rate irrigation type at boundary
    df["canal_at_boundary"] = (
        (df["Irrigation_Type"] == "Canal") &
        df["magic_score"].isin([0, 3])
    ).astype(np.int8)

    return df

# ============================================================
# SECTION 2 — LOAD DATA
# ============================================================
print("\n[1] Loading data...")
t0 = time.time()

train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)
orig_df   = pd.read_csv(ORIGINAL_PATH)

if "Irrigation_Requirement" in orig_df.columns:
    orig_df = orig_df.rename(columns={"Irrigation_Requirement": TARGET})

max_id = int(train_raw["id"].max())
orig_df["id"] = range(max_id + 1, max_id + 1 + len(orig_df))

n_competition = len(train_raw)

train_raw[TARGET] = train_raw[TARGET].map(TARGET_MAPPING)
orig_df[TARGET]   = orig_df[TARGET].map(TARGET_MAPPING)

train_full = pd.concat([train_raw, orig_df], ignore_index=True)

print(f"  Competition rows : {n_competition:,}")
print(f"  Original rows    : {len(orig_df):,}")
print(f"  Combined train   : {len(train_full):,}")
print(f"  Test rows        : {len(test_raw):,}")
print(f"  Loaded in {time.time()-t0:.1f}s")

# ============================================================
# SECTION 3 — FEATURE ENGINEERING
# ============================================================
print(f"\n[2] Engineering features (FE {FE_VERSION})...")
t0 = time.time()

train_eng = engineer_features(train_full)
test_eng  = engineer_features(test_raw)

print(f"  Train shape : {train_eng.shape}")
print(f"  Test shape  : {test_eng.shape}")
print(f"  Engineered in {time.time()-t0:.1f}s")

# ============================================================
# SECTION 4 — FEATURE VALIDATION
# ============================================================
print(f"\n[3] Validating features...")

# 4.1 All expected new features exist
missing = [f for f in NEW_FEATURES if f not in train_eng.columns]
if missing:
    raise ValueError(f"Missing features in train_eng: {missing}")
print(f"  ✅ All {len(NEW_FEATURES)} new features present")

# 4.2 No NaN in engineered features
nan_cols = [c for c in NEW_FEATURES
            if c in train_eng.columns and train_eng[c].isna().any()]
if nan_cols:
    print(f"  ⚠  NaN found in: {nan_cols}")
    print(f"     Filling with column median...")
    for c in nan_cols:
        train_eng[c] = train_eng[c].fillna(train_eng[c].median())
        test_eng[c]  = test_eng[c].fillna(train_eng[c].median())
else:
    print(f"  ✅ No NaN in new features")

# 4.3 Zero-variance check on hard examples
if os.path.exists(HARD_IDX_PATH):
    hard_idx  = np.load(HARD_IDX_PATH)
    hard_mask = np.zeros(n_competition, dtype=bool)
    hard_mask[hard_idx] = True

    train_comp = train_eng.iloc[:n_competition]
    hard_rows  = train_comp.iloc[hard_mask]

    num_feature_cols = [
        c for c in NEW_FEATURES
        if c in train_eng.columns
        and train_eng[c].dtype in [np.float32, np.float64,
                                    np.int8, np.int32, np.int64, "int64"]
    ]
    zero_var = [
        c for c in num_feature_cols
        if hard_rows[c].var() < 1e-10
    ]
    if zero_var:
        raise ValueError(
            f"\n❌ ZERO-VARIANCE ON HARD ROWS — DO NOT TRAIN:\n"
            + "\n".join(f"   - {c}" for c in zero_var)
            + "\n\nFix these features in engineer_features() before proceeding."
        )
    print(f"  ✅ Zero-variance check passed on {hard_mask.sum():,} hard rows")

    # 4.4 Print hard-row feature summary for key new features
    print(f"\n  Key new feature stats on hard rows vs all rows:")
    print(f"  {'Feature':<30} {'Hard mean':>10} {'All mean':>10} {'Hard std':>10}")
    print("  " + "─" * 65)
    check_cols = [
        "dual_boundary_proximity", "boundary_proximity",
        "prior_irrigation_intensity", "high_prior_irrigation",
        "margin_Medium_wins", "formula_predicts_medium",
        "humidity_at_boundary", "vpd_at_boundary",
    ]
    for col in check_cols:
        if col in train_eng.columns:
            h_mean = hard_rows[col].mean()
            a_mean = train_comp[col].mean()
            h_std  = hard_rows[col].std()
            print(f"  {col:<30} {h_mean:>10.4f} {a_mean:>10.4f} {h_std:>10.4f}")
else:
    print(f"  ⚠  Hard example indices not found at {HARD_IDX_PATH}")
    print(f"     Skipping zero-variance and hard-row checks")

# ============================================================
# SECTION 5 — SAVE PARQUET FILES
# ============================================================
print(f"\n[4] Saving engineered datasets...")
t0 = time.time()

train_path_out = f"{OUT_DIR}train_engineered_{FE_VERSION}.parquet"
test_path_out  = f"{OUT_DIR}test_engineered_{FE_VERSION}.parquet"

train_eng.to_parquet(train_path_out, index=False)
test_eng.to_parquet( test_path_out,  index=False)

print(f"  Saved train: {train_path_out}")
print(f"  Saved test : {test_path_out}")
print(f"  Saved in {time.time()-t0:.1f}s")

# ============================================================
# SECTION 6 — SAVE METADATA
# ============================================================
print(f"\n[5] Saving feature metadata...")

# MD5 checksums — model notebooks verify these to confirm correct file loaded
def md5(path):
    return hashlib.md5(open(path, "rb").read()).hexdigest()

# Column type classification for model notebooks
all_cols     = list(train_eng.columns)
object_cols  = [c for c in all_cols if train_eng[c].dtype == object]
numeric_cols = [c for c in all_cols
                if c not in object_cols + ["id", TARGET]]

metadata = {
    "fe_version"       : FE_VERSION,
    "created_at"       : time.strftime("%Y-%m-%d %H:%M:%S"),
    "n_competition"    : n_competition,
    "n_train_rows"     : len(train_eng),
    "n_test_rows"      : len(test_eng),
    "n_total_columns"  : len(train_eng.columns),
    "n_new_features"   : len(NEW_FEATURES),
    # Column lists
    "num_cols"         : NUMS,
    "cat_cols"         : CATS,
    "new_features"     : NEW_FEATURES,
    "all_numeric_cols" : numeric_cols,
    "target_col"       : TARGET,
    # Thresholds
    "soil_thresh"      : SOIL_THRESH,
    "rain_thresh"      : RAIN_THRESH,
    "temp_thresh"      : TEMP_THRESH,
    "wind_thresh"      : WIND_THRESH,
    # Checksums
    "train_checksum"   : md5(train_path_out),
    "test_checksum"    : md5(test_path_out),
    # Hard example info
    "hard_idx_path"    : HARD_IDX_PATH,
    "hard_idx_exists"  : os.path.exists(HARD_IDX_PATH),
    # Notes
    "removed_features" : [
        "score_High", "score_Low", "score_Medium",
        "margin_High_Medium", "margin_Low_Medium",
    ],
    "removal_reason"   : (
        "score_Medium has zero variance on hard rows (= 2.0 constant). "
        "score_High/Low are linear transforms of magic_score. "
        "margin_High/Low_Medium are redundant with magic_score."
    ),
}

meta_path = f"{OUT_DIR}feature_metadata_{FE_VERSION}.json"
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"  Saved metadata: {meta_path}")

# ============================================================
# SECTION 7 — SUMMARY
# ============================================================
print(f"\n{'='*60}")
print(f"Feature Engineering {FE_VERSION} — Complete")
print(f"{'='*60}")
print(f"  Train parquet : {train_path_out}")
print(f"  Test parquet  : {test_path_out}")
print(f"  Metadata      : {meta_path}")
print(f"  Train shape   : {train_eng.shape}")
print(f"  Test shape    : {test_eng.shape}")
print(f"  New features  : {len(NEW_FEATURES)}")
print(f"  FE version    : {FE_VERSION}")
print(f"\n  Checksums:")
print(f"    train: {metadata['train_checksum']}")
print(f"    test : {metadata['test_checksum']}")
print(f"\n  ✅ Ready for model training")
print(f"{'='*60}")

# ============================================================
# SECTION 8 — MODEL NOTEBOOK LOADER TEMPLATE
# ============================================================
print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
COPY THIS BLOCK INTO EACH MODEL NOTEBOOK
(replaces Sections 2-4 in xgb_v21.py, lgbm.py, catboost.py)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import json, hashlib
FE_VERSION = "v21"
OUT_DIR    = "/content/drive/MyDrive/irrigation_need_v15/"

# ── Load metadata ────────────────────────────────────────────
with open(f"{OUT_DIR}feature_metadata_{FE_VERSION}.json") as f:
    meta = json.load(f)
assert meta["fe_version"] == FE_VERSION, "FE version mismatch"

# ── Load parquet ─────────────────────────────────────────────
print(f"Loading pre-engineered features (FE {FE_VERSION})...")
train_eng = pd.read_parquet(f"{OUT_DIR}train_engineered_{FE_VERSION}.parquet")
test_eng  = pd.read_parquet(f"{OUT_DIR}test_engineered_{FE_VERSION}.parquet")

# ── Verify checksum ──────────────────────────────────────────
actual = hashlib.md5(
    open(f"{OUT_DIR}train_engineered_{FE_VERSION}.parquet","rb").read()
).hexdigest()
assert actual == meta["train_checksum"], f"Checksum mismatch: {actual}"

# ── Recover column lists ─────────────────────────────────────
NUMS         = meta["num_cols"]
CATS         = meta["cat_cols"]
NEW_FEATURES = meta["new_features"]
n_competition = meta["n_competition"]

print(f"  Train: {train_eng.shape}  Test: {test_eng.shape}")
print(f"  FE {FE_VERSION} loaded and verified ✅")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")